# Facial Keypoints Detection — v4a — training recipe: augmentation, temperature, joint supervision, flip test

Same backbone and 24x24 head as v2; only the recipe changes. Integral Regression (Sun et al.) trains with translation +-2%, scale +-25%, rotation +-30 deg and flip, pre-trains the heatmap to saturation, finds joint heatmap + coordinate supervision (I1) best for 2D, and reports a flip-test gain. Adaptive Wing (Wang, Bo, Li Fuxin; Oregon State University) adds blur, noise and occlusion augmentation. Measured here before training: v2's softmax temperature (beta=1 on 0-1 Gaussian logits) puts soft-argmax ~14px off even an ideal heatmap; sigma=2 with beta=12 brings that floor to 0.03px, so the temperature is part of the recipe.

Same 88% / 12% split as v2 (`random_state=42`), so the held-out RMSE is directly comparable to v2's 2.4385px. One log line per epoch goes to `logs/v4a.log`; the best weights are checkpointed on every improvement to `checkpoints/v4a.pt`. No figures are drawn during training: every epoch appends statistics (loss split, held-out RMSE overall and per coordinate, centre-pull slope, softmax entropy, temperature, gradient norm, learning rate) to `results/v4a_history.json`; they are plotted in the post-training notebook.

In [1]:
import json, time
import torch
import fkd_v4_common as C
print('device:', C.device(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
data = C.load_data('data')
print('train rows', len(data['idx_tr']), '| held-out rows', len(data['idx_va']))

device: cuda NVIDIA L4


train rows 6203 | held-out rows 846


In [2]:
RUN = 'v4a'
CFG = {'epochs': 240,
 'a_max': 40,
 'a_patience': 6,
 'batch': 512,
 'lr': 0.001,
 'lr_b': 0.001,
 'sched': 'step',
 'patience': 30,
 'seed': 42,
 'opt': 'adamw',
 'wd': 0.0001,
 'H': 24,
 'sigma': 2.0,
 'heatmap_loss': 'mse',
 'hm_frac': 0.2,
 'aug': {'affine': True, 'rot_deg': 30, 'scale': 0.2, 'translate': 0.06, 'photometric': True, 'cutout': True}}
model = C.SingleStageModel(H=24, n_deconv=1, beta=12.0)
print(sum(p.numel() for p in model.parameters()), 'parameters')

622032 parameters


## Training (Stage A heatmap warm-up to plateau, then Stage B) — statistics only, no figures

In [3]:
t0 = time.time()
model, history, best = C.train(RUN, model, data, CFG)
print(f'best held-out RMSE {best:.4f}px | wall {time.time()-t0:.0f}s')

2026-10-08T12:28:26Z [v4a] start | params=622,032 | cfg={"epochs": 240, "a_max": 40, "a_patience": 6, "batch": 512, "lr": 0.001, "lr_b": 0.001, "sched": "step", "patience": 30, "seed": 42, "opt": "adamw", "wd": 0.0001, "H": 24, "sigma": 2.0, "heatmap_loss": "mse", "hm_frac": 0.2, "aug": {"affine": true, "rot_deg": 30, "scale": 0.2, "translate": 0.06, "photometric": true, "cutout": true}}


fkd_v4_common.py:510: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  k = len(idx); s["loss"] += float(loss) * k; s["l1"] += float(l1) * k; s["hm"] += float(ht) * k; s["n"] += k


2026-10-08T12:28:59Z [v4a:A] epoch   0 train_hm 0.15147 val_hm 0.02458 val_rmse 17.6666 beta 12.00 29.9s


2026-10-08T12:29:05Z [v4a:A] epoch   1 train_hm 0.02589 val_hm 0.02045 val_rmse 17.7267 beta 12.00 5.3s


2026-10-08T12:29:10Z [v4a:A] epoch   2 train_hm 0.02112 val_hm 0.01990 val_rmse 17.6352 beta 12.00 4.9s


...

2026-10-08T12:45:42Z [v4a:B1] epoch 228 loss 2.4494 (l1 1.6768 hm 0.006249) val_rmse 2.3187 best 2.3055 slope 0.798 lr 1.0e-05 beta 18.73 gnorm 10.18 3.2s


2026-10-08T12:45:45Z [v4a:B1] epoch 229 loss 2.4618 (l1 1.6897 hm 0.006246) val_rmse 2.3165 best 2.3055 slope 0.797 lr 1.0e-05 beta 18.73 gnorm 7.62 3.2s


2026-10-08T12:45:45Z [v4a:B1] early stop at epoch 229 (no improvement for 30 epochs)


2026-10-08T12:45:45Z [v4a] done | best held-out RMSE 2.3055 px | epochs used 230


best held-out RMSE 2.3055px | wall 1040s


## Held-out evaluation: decoders x flip test, per-keypoint RMSE, centre-pull slopes

In [4]:
res = C.evaluate_and_save(RUN, model, data, CFG, decoders=('softargmax', 'argmax_shift', 'dark'))
for k, v in res.items():
    if k.startswith('rmse') or k in ('shrinkage_slope_mean', 'beta_final'):
        print(f'{k:28s} {v}')
print('v2 reference 2.4385 px | target 1.9 px')

rmse_softargmax              2.305520534515381
rmse_softargmax_tta          2.1979854106903076
rmse_argmax_shift            2.930635690689087
rmse_argmax_shift_tta        2.5704855918884277
rmse_dark                    2.571990489959717
rmse_dark_tta                2.382594347000122
shrinkage_slope_mean         0.7957021822069965
beta_final                   18.715999603271484
v2 reference 2.4385 px | target 1.9 px


## Test-set submission (written, not submitted)
Decoder chosen by held-out RMSE with flip test among the three evaluated above (a selection over three options, stated as such).

In [5]:
cands = {d: res[f'rmse_{d}_tta'] for d in ('softargmax', 'argmax_shift', 'dark')}
best_dec = min(cands, key=cands.get)
n = C.write_submission(RUN, model, data, CFG, tta=True, decoder=best_dec)
print('decoder', best_dec, '| held-out', round(cands[best_dec], 4), '| rows', n)

decoder softargmax | held-out 2.198 | rows 27124
